# Classification using tree models - skeleton

Dataset: HeartFailurePrediction_data

Some remarks:

- Throughout the cases, multiple bold __questions__ are stated. The goal of these questions is to guide you through the  case and to help you interpret results and steps. There are also more challenging __deepdive questions__.
- This data file has been modified for the purposes of this case <br>
- This file offers some directions one can choose from when building a model. Different choices within the modelling process can lead to different models, which are not necessarily better or worse.

**Content:**
1. [Install packages](#1)
1. [Load data](#2)
1. [Data exploration](#3)
1. [Data preparation](#4)
1. [Train and validate models](#5)
    1. [Decision Tree CART](#5a) 
    1. [Random Forest](#5b)
    1. [AdaBoost](#5c)
    1. [XGBoost](#5d)
    1. [LightGBM](#5e)
    1. [CatBoost](#5f)
    1. [Comparison of Models](#5g)
1. [SHAP](#6)   
1. [K-fold cross-validation](#7)

In [ ]:
# Import packages
# 'as' := we abbreviate the package for common use
import os                                               # Set working directory            
import pandas as pd                                     # Data analysis tools               
import numpy as np                                      # Scientific computing for (multi-dimensional) arrays and matrices   
import matplotlib.pyplot as plt                         # 2D plotting library 
import seaborn as sns                                   # Data visualization library, extension of matplotlib  
import random                                           # Randomly generating numbers
import math                                             # Mathematical functions                  
import scipy.stats as stats                             # Fit of distribution plot
from sklearn.model_selection import train_test_split    # Split train-/testset     
from sklearn.tree import DecisionTreeClassifier         # Modeling CART Decision Tree  
from sklearn import metrics                             # Performance metrics
from sklearn.metrics import accuracy_score              # Accuracy of a model
from sklearn.metrics import classification_report       # Performance report of a classification model
from sklearn import tree                                # Visualizing tree 
from sklearn.ensemble import RandomForestClassifier     # Modeling Random Forest
from sklearn.ensemble import AdaBoostClassifier         # Modeling AdaBoost 
import xgboost as xgb                                   # Modeling XGBoost
from sklearn.model_selection import KFold               # Cross-validation using stratified K-fold



<a id="2"></a> 
## 2. Load data

In [ ]:
# a) Load data file
#    Note that all decimals should be . to be read in as floats. If not, they will be read as strings.
file = "..\\data\\HeartFailurePrediction.xlsx"
inputdata = pd.read_excel(file, header=0)
# b) Save an original copy of the data
inputdata_org = inputdata.copy()    

# c) Show the first lines of the data
inputdata.head()  

<a id="3"></a> 
## 3. Data exploration

In [ ]:
# a) Get an overview of the data
print('(nrow, ncol):', inputdata.shape)     # Get the number of rows and columns

In [ ]:
# b) Check data types of all variables
inputdata.dtypes

In [ ]:
# c) Univariate analysis
#    i. Show a brief summary of the variables
inputdata.describe()                 # min/max, count, mean, std and percentiles

__Question:__ when looking at the statistics presented above, are there any reasons to doubt the validity of all observations?

In [ ]:
#    ii. Get an overview of the NULLS in the dataset
nulls = inputdata.isnull().sum()                          # Number of NULLS 
percNulls = inputdata.isnull().sum()/inputdata.shape[0]   # Number of NULLS as a percentage

pd.concat({"# NULLS": nulls, "% NULLS": percNulls*100}, axis=1)

In [ ]:
# iii. Delete all entries with missing variables, and verify that there are indeed no more zero-entries

inputdata = inputdata.dropna()

nulls = inputdata.isnull().sum()                          # Number of NULLS 
percNulls = inputdata.isnull().sum()/inputdata.shape[0]   # Number of NULLS as a percentage

pd.concat({"# NULLS": nulls, "% NULLS": percNulls*100}, axis=1)

As you can see, deleting observations is the easiest way to tackle the issue of missing variables. However, this is not always the best way to do so.

__Deepdive question:__ under what circumstances is it okay to delete observations as a way to tackle missing variables? Moreover, what could be a better way to deal with the issue?

In [ ]:
#    iv. Create boxplot
# Adjust the line below to also find boxplots for other numerical variables.
x = ...
plt.figure(figsize=(7.5,5))
sns.boxplot(x=inputdata[x])
plt.title("Boxplot of " + x)
plt.xlabel(x)
plt.show()

In [ ]:
#    v. Create histogram 
x = ...
num_bins = 20
plt.figure(figsize=(7.5,5))
plt.hist(inputdata[x], num_bins)
plt.title("Histrogram plot of " + x + " with " + str(num_bins) + " bins")
plt.xlabel(x)
plt.show()

In [ ]:
# vi. Delete all zero-values for Cholesterol and create a new histogram
# A cholesterol-value of zero does not make any sense. So, we delete all entries where cholesterol equals zero.
inputdata = ...
x = ...
num_bins = ...
plt.figure(figsize=(7.5,5))
plt.hist(inputdata[x], num_bins)
plt.title("Histrogram plot of " + x + " with " + str(num_bins) + " bins")
plt.xlabel(x)
plt.show()                      

Deleting observations where the value for Cholesterol is zero is the easiest way to solve this issue.

__Deepdive question:__ we again decide to delete observations that have a Cholesterol value of zero. However, in this case, this is the case for a substantial part of observations. What is (especially in this case) the risk of deciding to 'just delete' these observations? Furthermore, how can we check if we can validly do so? Lastly, if we can't, what would be a better way to deal with the issue?

In [ ]:
#    vii. Create distribution plot
x = ...

plt.figure(figsize=(7.5,5))
sns.displot(data=inputdata[~np.isnan(inputdata[x])][x], stat='density', bins=..., kde=True)
plt.title("Distribution plot of " + x) 
...

In [ ]:
# d) Multivariate analysis
#    i. Count frequency of a variable
y = ...
x = ...
number = inputdata[x].value_counts()            # Get number per 'Count_variable'
average = inputdata.groupby(x)[y].mean()        # Get average of variable per 'Count_variable'           

pd.concat({"Number of " + x: number, "Mean of " + y + " per " + x: average}, axis=1)

In [ ]:
#    iii. Create distribution plot for two variables
y = ...
x = ...
...
sns.displot(data=..., x=..., hue=..., kind=..., label=...)
plt.title(...)
plt.xlabel(x)
plt.show()

__Question:__ what information does this graph reveal? Is it in line with your intuition?

In [ ]:
#    iv. Create scatterplot for two variables
y = ...
x = ...
plt.scatter(inputdata[x], inputdata[y], marker='o')
plt.title(...)
...

__Deepdive question:__ notice that there seems to be some correlation between RestingBP (resting blood pressure) and MaxHR (maximum heart rate achieved). How would you test whether this is indeed the case? Moreover, if there turns out to be a correlation between the two, does this correlation also imply that there is a causal effect? Why (not)? What factor could also play a role here?

In [ ]:
#    v. Create barplot for two variables
...
sns.barplot(x=inputdata[x], y=inputdata[y])
...

__Deepdive question:__ according to this graph, women have a higher Cholesterol level than men. What else should we check before we come to this conclusion for the general population? Moreover, how would we take this fact into account when building our models?

In [ ]:
#    vi. Create barplot for three variables
...
sns.barplot(data=inputdata, x=inputdata[...], y=inputdata[...], hue=inputdata[...],)
...

In [ ]:
#    vii. Create boxplot for two variables
...
sns.boxplot(...)
...

In [ ]:
#    viii. Create boxplot for three variables
...
sns.boxplot(...)
...

In [ ]:
# e) Correlation matrix
corrmat = inputdata.loc[:,[...]].corr()
f, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corrmat, vmax=.8, square=True, annot=True, cmap='RdBu_r')

__Deepdive question:__ how can this correlation matrix help us build a good model? For example, what variable seems to have most predictive power for heart diseases? Are there any multicollineary effects that we might consider taking into account?

<a id="4"></a> 
## 4. Data preparation

In [ ]:
# a) Handling outliers
x = 'Age'
...
sns.displot(...)
...

#    Fill the outlier values with the mean, median, most common value, maximum or minimum
outlier = 'Age'
mode_outlier = ...
inputdata[outlier][inputdata[outlier]> ...] = mode_outlier.iloc[0]

__Question:__ what could be a downside of replacing cells with outliers with the mean, when the outliers are extremely high values?

__Question:__ what is a good way to identify outliers? In other words, how do you decide the cut-off value for an observation to be an outlier?

In [ ]:
# b) Dummify variables
to_dummify = [...           # ADD ALL VARIABLES YOU WISH TO DUMMIFY 

             ]
dummified_columns = inputdata[to_dummify]                      # Save the original columns
inputdata = pd.get_dummies(inputdata, columns=to_dummify)      # Dummify the columns
inputdata = pd.concat([inputdata, dummified_columns], axis=1)  # Join the original columns to the new dataframe

Now, different variables have been turned into dummy variables. For example, 'Sex' has been split up into 'Sex_M' and 'Sex_F', to which the value 1 is assigned is an observation is male or female respectively.

__Question:__ what should we definetely take into account when we want to use these dummy variables when building our model?

#### Feature engineering

You can achieve a lot by creating new features and implementing these in the models. Can you improve the classification models by creating and adding new features?

In [ ]:
...

As you can see, as an example, we have added two new variables. 

1. We have added an interaction term between the dummy 'Sex_M' and 'RestingBP'. This is because resting blood pressure differs for men and women, according to the literature. 
2. We have also added a new dummy variable that takes the value '1' when Cholesterol is especially high (>350).

__Deepdive question:__ what other features can you think of in the feature engineering phase that might improve the model?

#### Split into train- and testset

In [ ]:
data_train, data_test  = ...

__Question__: note that we only spilt the data set into two parts after we did the different explorative analyses. According to some, it is better to make the split before the data exporation. Can you think of a reason why?

<a id="5"></a> 
## 5. Train and validate models

<a id="5a"></a> 
### A) Decision Tree CART

In [ ]:
# Define X and y 
X_variables = [...
               # ADJUST VARIABLES TO THOSE YOU WISH TO INCLUDE
              ]
y_variable = ...

X_train = data_train.loc[:, X_variables]
y_train = data_train[y_variable].astype('int')
X_test = data_test.loc[:, X_variables]
y_test = data_test[y_variable].astype('int')

In [ ]:
# a) Set model parameters 
#    Note: if Min_bucket is too large, the tree might not branch, if too small, the tree might get to big to interpret
Min_num_splits = ...                           # Minimum number of items to split
Min_bucket     = ...                          # Minimum number of items per bucket
Max_depth      = ...                           # Maximum depth of final tree (nr of levels)

__Question:__ how does the model change when you increase the min_num_splits? How about if you decrease max_depth?

__Question:__ discuss what you think are the best values are for these parameters.

In [ ]:
# b) Estimate the model 
mytree = DecisionTreeClassifier(max_depth=Max_depth
                                ,min_samples_split=Min_num_splits
                                ,min_samples_leaf=Min_bucket
                                ,criterion="gini"              
                                ,splitter="best"
                                ,random_state=random.seed()
                                )

mytree.fit(X_train, y_train)     # Fit the model over the train set

In [ ]:
# c) Create predictions for the test set
preds_proba = mytree.predict_proba(X_test)
preds = mytree.predict(X_test)   # Cut-off point equals 0.5

#    Show the first 5 lines of the prediction probabilities and corresponding prediction
pd.concat([pd.DataFrame(preds_proba, columns=["Prob. 0", "Prob. 1"]), pd.DataFrame(preds, columns=["Prediction"])], axis=1).head()

In [ ]:
#     Calculate the optimal cut-off point given these theoretical costs.

cost_TP = 0
cost_TN = 0
cost_FP = 50
cost_FN = 100
total_cost = math.inf

for i in np.linspace(0,1,100,endpoint=False):
    y_pred = (preds_proba[:,1]>i).astype('int')
    results = metrics.confusion_matrix(y_pred,y_test)
    TN = results[0][0]
    FN = results[0][1]
    FP = results[1][0]
    TP = results[1][1]
    
    # Calculate cutoff-point
    cost = ...
    total_cost = min(total_cost,cost)
    if(total_cost == cost):
        opt_cutoff = i
        
print('Optimal cut-off:', opt_cutoff)

#    Create predictions for the test set according to optimal cutoff point
preds = (preds_proba[:,1] > opt_cutoff).astype('int')

__Question:__ How would the results change if the cost of False Positives (cost_FP) increased to 100?

__Question:__ What could be a reason for the costs of false negatives to be higher than the costs of false positives in this situation? Can you come up with a situation in which the relation it is the other way around?

In [ ]:
# d) Evaluate results
#    i. Create confusion matrix
print(pd.crosstab(preds, y_test))

In [ ]:
#    ii. Create classification report
print(classification_report(y_test, preds))

__Question:__ what do you think when you see the results? Does the model perform as hoped / expected?

In [ ]:
#    iii. Get the feature importances of the tree
importances = mytree.feature_importances_ 
std = np.std([mytree.feature_importances_], axis=0)
indices = np.argsort(importances)[::-1]

importances_features = []
print("Feature ranking:")                    # Print the feature ranking
for f in range(X_train.shape[1]):
    print("Feature %d (%s) %f" % (indices[f], X_variables[indices[f]], importances[indices[f]]))
    importances_features.append(X_variables[indices[f]])

plt.figure(figsize=(7.5,5))
plt.figure()                                 # Plot the feature importances 
plt.bar(range(X_train.shape[1]), importances[indices],
       color="r", yerr=std[indices], align="center")
plt.title("Feature importances")
plt.ylabel("Importance in terms of decreasing the weighted impurity")
plt.xlabel("Feature")
plt.xticks(range(X_train.shape[1]), importances_features, rotation = 30)
plt.xlim([-1, X_train.shape[1]])
plt.show()

__Question:__ How would you explain this graph to a non analytical expert?

In [ ]:
#    iv. Create ROC curve
fpr, tpr, t = metrics.roc_curve(y_test, preds_proba[:,1])

#     v. Calculate AUC
roc_auc_tree = metrics.auc(fpr, tpr)

plt.figure(figsize=(10,6))
plt.plot([0,1],[0,1],linestyle = '--',lw = 2,color = 'black')      # Plot results
plt.plot(fpr, tpr, lw=2, alpha=0.3, label='Mean ROC Decision Tree CART (AUC = %0.2f)' % (roc_auc_tree))
plt.title('ROC curve')
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.legend(loc="lower right")
plt.show()

__Question:__ what does this ROC curve imply for how accurate your diagnosis is?

In [ ]:
#    Plot tpr vs 1-fpr
i = np.arange(len(tpr)) # index for df
roc = pd.DataFrame({'fpr' : pd.Series(fpr, index=i),'tpr' : pd.Series(tpr, index = i)
                    ,'1-fpr' : pd.Series(1-fpr, index = i)
                    ,'tf' : pd.Series(tpr - (1-fpr), index = i)
                    ,'thresholds' : pd.Series(t, index = i)})
print(roc.loc[(roc.tf-0).abs().argsort()[:1]])
fig, ax = plt.subplots()
plt.plot(roc['tpr'])
plt.plot(roc['1-fpr'], color = 'red')
plt.title('Receiver operating characteristic')
plt.ylabel('True Positive Rate')
plt.xlabel('1-False Positive Rate')
ax.set_xticklabels([])

__Question:__ what information does this graph portray? What is the meaning of the intersection between the red and the blue line? 

In [ ]:
# Visualize Decision Tree

plt.figure(figsize=(20, 10))  # width=20, height=10 inches
tree.plot_tree(mytree, feature_names=X_variables, filled=True, fontsize=14)
plt.show()

<a id="5c"></a> 
### B) Random Forest 

In [ ]:
# Define X and y 
X_variables = ...
y_variable = ...

X_train = ...
y_train = ...
X_test = ...
y_test = ...

In [ ]:
# a) Set model parameters 
#     Note: You can tweak the size of your forest by changing 'N_trees', but note computation time increases with size
#     Note: if Min_bucket is too large, the trees might not branch
N_trees        = ...                           # Number of trees that are estimated
Min_num_splits = ...                           # Minimum number of items to split    
Min_bucket     = ...                           # Minimum number of items per bucket
Max_depth      = ...                           # Maximum depth of each tree (nr of levels)

In [ ]:
# b) Estimate the model
forest = RandomForestClassifier(n_estimators = N_trees
                                ,criterion = "gini"            
                                ,max_depth = Max_depth
                                ,min_samples_split = Min_num_splits
                                ,min_samples_leaf = Min_bucket
                                ,random_state = random.seed()
                                )

forest.fit(...)   # Fit the model over the train set

In [ ]:
# c) Create predictions for the test set
preds_proba = ...
preds = ...

#    Show the first 5 lines of the prediction probabilities and corresponding prediction
pd.concat([pd.DataFrame(preds_proba, columns=["Prob. 0", "Prob. 1"]), pd.DataFrame(preds, columns=["Prediction"])], axis=1).head()

In [ ]:
# d) Evaluate results
#    i. Create confusion matrix
print(pd.crosstab(...))

In [ ]:
#    ii. Create classification report
print(classification_report(...))

In [ ]:
#    iii. Get the feature importances of the forest
importances = ...
std = ...
indices = ...

importances_features = []
print("Feature ranking:")                    # Print the feature ranking
for f in range(X_train.shape[1]):
    ...

plt.figure(figsize=(7.5,5))
plt.figure()                                 # Plot the feature importances 
plt.bar(range(X_train.shape[1]), importances[indices],
       color="r", yerr=std[indices], align="center")
plt.title("Feature importances")
plt.ylabel("Importance in terms of decreasing the weighted impurity")
plt.xlabel("Feature")
plt.xticks(range(X_train.shape[1]), importances_features, rotation = 30)
plt.xlim([-1, X_train.shape[1]])
plt.show()

In [ ]:
#    iv. Create ROC curve
fpr, tpr, t = ...

#     v. Calculate AUC
roc_auc_forest = metrics.auc(fpr, tpr)

plt.figure(figsize=(10,6))
plt.plot([0,1],[0,1],linestyle = '--',lw = 2,color = 'black')      # Plot results
plt.plot(fpr, tpr, lw=2, alpha=0.3, label='Mean ROC Random Forest (AUC = %0.2f)' % (roc_auc_forest))
plt.title(...)
plt.ylabel(...)
plt.xlabel(...)
plt.legend(loc="lower right")
plt.show()

<a id="5d"></a> 
### C) ADABOOST (Adaptive Boosting Model)

In [ ]:
# Define X and y 
X_variables = ...'

X_train = ...

In [ ]:
# a) Set model parameters 
#     Note: You can tweak the size of your boost by changing 'N_trees', but note computation time increases with size
#     Note: if Min_bucket is too large, the trees might not branch
N_trees        = 1000                           # Number of trees that are estimated
Min_num_splits = 4                           # Minimum number of items to split    
Min_bucket     = 3                           # Minimum number of items per bucket
Max_depth      = 3                           # Maximum depth of each tree (nr of levels)
Learning_rate  = 1                           # The learning rate

In [ ]:
# b) Estimate model
adaptboost = AdaBoostClassifier(n_estimators=N_trees
                                ,base_estimator=DecisionTreeClassifier(max_depth = Max_depth
                                                                       ,min_samples_split = Min_num_splits
                                                                       ,min_samples_leaf = Min_bucket
                                                                       ,criterion = "gini"              
                                                                       ,splitter = "best"
                                                                       )
                                ,learning_rate=Learning_rate
                               )

adaptboost.fit(X_train, y_train)      # Fit the model over the train set

In [ ]:
# c) Create predictions for the test set
preds_proba = ...
preds = ...

#    Show the first 5 lines of the prediction probabilities and corresponding prediction
...

In [ ]:
# d) Evaluate results
#    i. Create confusion matrix
print(pd.crosstab(...))

In [ ]:
#    ii. Create classification report
...

In [ ]:
#    iii. Get the feature importances of the Adaboost
importances = ...
std = ...
indices = ...

importances_features = []
print("Feature ranking:")                    # Print the feature ranking
for f in range(X_train.shape[1]):
    ...

...                                # Plot the feature importances 
plt.bar(range(X_train.shape[1]), importances[indices],
       color="r", yerr=std[indices], align="center")
plt.title(...)
plt.ylabel(...)
plt.xlabel(...)
plt.xticks(...)
plt.xlim(...)
plt.show()

In [ ]:
#    iv. Create ROC curve
...

#     v. Calculate AUC
roc_auc_adaboost = ...

plt.figure(...)
plt.plot(...)      # Plot results
plt.plot(...)
plt.title(...)
plt.ylabel(...)
plt.xlabel(...)
plt.legend(...)
plt.show()

In [ ]:
# e) Plot error with respect to number of estimators (trees)
trained_adaptboost = adaptboost.fit(X_train, y_train)  

real_test_errors = []
for real_test_predict in trained_adaptboost.staged_predict(X_test):
    real_test_errors.append(1. - accuracy_score(real_test_predict, np.ravel(y_test)))
n_trees_real = len(trained_adaptboost)
real_estimator_errors = trained_adaptboost.estimator_errors_[:n_trees_real]
plt.figure(figsize=(15, 5))

plt.plot(range(1, n_trees_real + 1),
         real_test_errors, c='black',
         linestyle='dashed')
plt.ylim(min(real_test_errors), max(real_test_errors)+0.1)
plt.title(...)
plt.ylabel(...)
plt.xlabel(...)

<a id="5e"></a> 
### D) XGBoost

In [ ]:
# Define X and y 
...

In [ ]:
# a) Set model parameters 
N_trees        = ...                          # Number of trees that are estimated
Max_depth      = ...                           # Maximum depth of each tree (nr of levels)
Learning_rate  = ...                           # The learning rate ('eta')
Min_bucket     = ...                           # Minimum number of items per bucket
Subsample      = ...                          # Subsample ratio of the training instance
Silent         = ...                          # Whether to print messages while running boosting

In [ ]:
# b) Estimate model
XGB = xgb.XGBClassifier(n_estimators=N_trees
                        ,max_depth=Max_depth 
                        ,learning_rate=Learning_rate
                        ,min_child_weight=Min_bucket
                        ,subsample=Subsample
                        ,silent=Silent
                       )

...      # Fit the model over the train set

In [ ]:
# c) Create predictions for the test set
...

In [ ]:
# d) Evaluate results
#    i. Create confusion matrix
...

In [ ]:
#    ii. Create classification report
...

In [ ]:
#    iii. Get the feature importances of the XGBoost
...

In [ ]:
#    iv. Create ROC curve
...

#     v. Calculate AUC
...

In [ ]:
# e) Plot error with respect to number of estimators (trees)
eval_set = ...
eval_metric = ...
trained_xgboost = XGB.fit(...)  

real_test_predict = ...
predictions = ...

# retrieve performance metrics
results = ...
epochs = ...
x_axis = ...

fig, ax = ...
ax.plot(...)
ax.plot(...)
ax.legend()
...

<a id="5e"></a> 
### E) LightGBM

In [ ]:
# Import relevant package
import lightgbm as lgb
import time
from sklearn.metrics import f1_score,average_precision_score                    # f1 score of model

In [ ]:
# Define X and y 
# X_variables = [''                 # ADJUST VARIABLES TO THOSE YOU WISH TO INCLUDE

#               ]
# y_variable = ''

X_train = data_train.loc[:, X_variables]
y_train = data_train[y_variable]
X_test = data_test.loc[:, X_variables]
y_test = data_test[y_variable]

print(X_variables)

In [ ]:
# a) Set model parameters 
N_trees           = ...                           # Number of trees that are estimated
Max_depth         = ...# Maximum depth of each tree (nr of levels); -1 means no limit
Learning_rate     = ...                           # The learning rate ('eta')
Min_child_samples = ...                            # Minimum number of items per bucket
Min_child_weight  = ...                         # Minimum sum of instance weight (hessian) needed in a child (leaf).
Subsample         = ...                             # Subsample ratio of the training instance
Silent            = ...                             # Whether to print messages while running boosting
Num_leaves        = ...                            # Maximum number of leafs per tree

# Note: there are way more hyperparameters to tune. You can find them well-explained here: 
# https://lightgbm.readthedocs.io/en/latest/Parameters-Tuning.html
# The values above are not tuned and optimized yet

In [ ]:
# b) Estimate model
lgb_model = lgb.LGBMClassifier(
                         n_estimators=N_trees
                        ,max_depth=Max_depth 
                        ,learning_rate=Learning_rate
                        ,min_child_weight=Min_child_weight
                        ,min_child_samples=Min_child_samples
                        ,subsample=Subsample
                        ,num_leaves=Num_leaves
                        ,silent=Silent
                       )

lgb_model.fit(X_train, y_train)      # Fit the model over the train set

In [ ]:
# c) Create predictions for the test set
preds_proba = lgb_model.predict_proba(X_test)
preds = lgb_model.predict(X_test)   # Cut-off point equals 0.5

# Show the first 5 lines of the prediction probabilities and corresponding prediction
pd.concat([pd.DataFrame(preds_proba, columns=["Prob. 0", "Prob. 1"]), pd.DataFrame(preds, columns=["Prediction"])], axis=1)

In [ ]:
# d) Evaluate results
#    i. Create confusion matrix
print(pd.crosstab(preds, y_test))

In [ ]:
#    ii. Create classification report
#print(classification_report(y_test, preds))
f1_lgb_model = f1_score(y_test, preds)

In [ ]:
#    iii. Get the feature importances of the lgb_modeloost
importances = lgb_model.feature_importances_ 
std = np.std([lgb_model.feature_importances_], axis=0)
indices = np.argsort(importances)[::-1]

importances_features = []
print("Feature ranking:")                    # Print the feature ranking
for f in range(X_train.shape[1]):
    print("Feature %d (%s) %f" % (indices[f], X_variables[indices[f]], importances[indices[f]]))
    importances_features.append(X_variables[indices[f]])


plt.figure(figsize=(7.5,5))
plt.figure()                                 # Plot the feature importances 
plt.bar(range(X_train.shape[1]), importances[indices],
        color="r", yerr=std[indices], align="center")
plt.title("Feature importances")
plt.ylabel("Importance in terms of decreasing the weighted impurity")
plt.xlabel("Feature")
plt.xticks(range(X_train.shape[1]), importances_features, rotation = 30)
plt.xlim([-1, X_train.shape[1]])
plt.show()

In [ ]:
#    iv. Create ROC curve
fpr, tpr, t = metrics.roc_curve(y_test, preds_proba[:,1])

#     v. Calculate AUC
lgb_model_roc_auc_tree = metrics.auc(fpr, tpr)

plt.figure(figsize=(10,6))
plt.plot([0,1],[0,1],linestyle = '--',lw = 2,color = 'black')      # Plot results
plt.plot(fpr, tpr, lw=2, alpha=0.3, label='Mean ROC Decision Tree LightGBM (AUC = %0.2f)' % (lgb_model_roc_auc_tree))
plt.title('ROC curve')
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.legend(loc="lower right")
plt.show()

<a id="5f"></a> 
### F) CatBoost

In [ ]:
# Import relevant package
import catboost as cat
import time

In [ ]:
# Define X and y 
X_variables = ['Age', 'RestingBP', 'Cholesterol', '...']


X_train = data_train.loc[:, X_variables]
y_train = data_train[y_variable]
X_test = data_test.loc[:, X_variables]
y_test = data_test[y_variable]

In [ ]:
# Because we use CatBoost, we can leverage its abilitiy to handle categoricals in a clever way
# For that, we need the column indices of the cat_features
cat_features=[X_variables.index("...")]

In [ ]:
# a) Set model parameters 
N_trees        = ...                           # Number of trees that are estimated
Max_depth      = ...                             # Maximum depth of each tree (nr of levels)
Learning_rate  = ...                             # The learning rate ('eta')
Subsample      = ...                             # Subsample ratio of the training instance
Silent         = True                          # Whether to print messages while running boosting

# Note: there are way more hyperparameters to tune. You can find them well-explained here: 
# https://catboost.ai/docs/concepts/parameter-tuning.html
# The values above are not tuned and optimized yet

In [ ]:
# b) Estimate model
cat_model = cat.CatBoostClassifier(
                        n_estimators=N_trees
                        ,max_depth=Max_depth 
                        ,learning_rate=Learning_rate
                        ,subsample=Subsample
                        ,silent=Silent
                        ,cat_features=cat_features 
                       )
# You can uncomment the cat_features if you don't want to use this. 
# Given that we don't have many categories, this should not matter much. 
# But we leave it up to you to find out what the difference in speed and performance is!

cat_model.fit(X_train, y_train)      # Fit the model over the train set

In [ ]:
# c) Create predictions for the test set
preds_proba = cat_model.predict_proba(X_test)
preds = cat_model.predict(X_test)   # Cut-off point equals 0.5

#    Show the first 5 lines of the prediction probabilities and corresponding prediction
pd.concat([pd.DataFrame(preds_proba, columns=["Prob. 0", "Prob. 1"]), pd.DataFrame(preds, columns=["Prediction"])], axis=1).head()

In [ ]:
# d) Evaluate results
#    i. Create confusion matrix
print(pd.crosstab(preds, y_test))

In [ ]:
#    ii. Create classification report
#print(classification_report(y_test, preds))
f1_cat_model = f1_score(y_test, preds)

In [ ]:
#    iii. Get the feature importances of the cat_modeloost
importances = cat_model.feature_importances_ 
std = np.std([cat_model.feature_importances_], axis=0)
indices = np.argsort(importances)[::-1]


importances_features = []
print("Feature ranking:")                    # Print the feature ranking
for f in range(X_train.shape[1]):
    print("Feature %d (%s) %f" % (indices[f], X_variables[indices[f]], importances[indices[f]]))
    importances_features.append(X_variables[indices[f]])


plt.figure(figsize=(7.5,5))
plt.figure()                                 # Plot the feature importances 
plt.bar(range(X_train.shape[1]), importances[indices],
        color="r", yerr=std[indices], align="center")
plt.title("Feature importances")
plt.ylabel("Importance in terms of decreasing the weighted impurity")
plt.xlabel("Feature")
plt.xticks(range(X_train.shape[1]), importances_features, rotation = 30)
plt.xlim([-1, X_train.shape[1]])
plt.show()

In [ ]:
#    iv. Create ROC curve
fpr, tpr, t = metrics.roc_curve(y_test, preds_proba[:,1])

#    v. Calculate AUC
cat_model_roc_auc_tree = metrics.auc(fpr, tpr)

plt.figure(figsize=(10,6))
plt.plot([0,1],[0,1],linestyle = '--',lw = 2,color = 'black')      # Plot results
plt.plot(fpr, tpr, lw=2, alpha=0.3, label='Mean ROC Decision Tree CatBoost (AUC = %0.2f)' % (cat_model_roc_auc_tree))
plt.title('ROC curve')
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.legend(loc="lower right")
plt.show()

<a id="5g"></a> 
### G) Comparison of Models

In [ ]:
print('AUC Decision Tree: CART', roc_auc_tree)
print('AUC Random Forest:', ...)
print(...)
print(...)
print(...)
print(...)

<a id="6"></a> 
## 6. SHAP

In [ ]:
import shap

In [ ]:
# Initialize X and y as in model under investigation
X_variables = [...
               # ADJUST VARIABLES TO THOSE YOU WISH TO INCLUDE
              ]
y_variable = 'HeartDisease'

X_train = data_train.loc[:, X_variables]
y_train = data_train[y_variable]
X_test = data_test.loc[:, X_variables]
y_test = data_test[y_variable]

In [ ]:
# Initialize shapley values for Random Forest with set X_test
explainer = shap.TreeExplainer(forest)
shap_values = explainer.shap_values(X_test)
preds = forest.predict(X_test)

In [ ]:
# Check for indices with predictions = 1
X_test[preds==1]

In [ ]:
# Select the shap values for predictions = 1
shap_values = shap_values[1]

In [ ]:
# Initialize JavaScript (necessary for Shap plot)
shap.initjs()

# Shapley plot for observation with prediction = 1
observation_index = ... #Choose index that has preds = 1
shap.force_plot(explainer.expected_value[1], shap_values[[i for i, x in enumerate(X_test.index==observation_index) if x][0],:], X_test.loc[observation_index, :])

In [ ]:
# Showing dependency on cholesterol variable
shap.dependence_plot('Cholesterol', shap_values, X_test, xmax="percentile(99)")

In [ ]:
# Showing dependency on all variables at once
shap.summary_plot(shap_values, X_test, show=False)
plt.gcf().axes[-1].set_aspect(100)
plt.gcf().axes[-1].set_box_aspect(100)

<a id="7"></a> 
## 7. Extra: K-fold cross validation

This last part is extra, on top of what you learned during class. You can do this when you understood all parts of the case and you still have time left. 

The idea behind K-fold cross validation is the following: note that the split between the training set and the test set is random. In order to correct for this randomness, we split the dataset into 'K' subsets, instead of only two. Next, we train the model K times, where each time, one of the K subsets functions as test set, and all the other subsets as training set. We then obtain the model paramteters by taking the average over K different estimations. 

In [ ]:
# Define X and y 
X_variables = ...
y_variable = ...

X = ...
y = ...

In [ ]:
i = 0

# Define the number of splits and create dataframe for saving AUC's
N_splits = 10
aucs = pd.DataFrame(index=range(0,N_splits), columns=['Decision Tree','Random Forest','Adaboost','XGBoost'])

# Iterate models using K-fold cross validation for indexing train- and test set
kf = KFold(n_splits=N_splits, shuffle=False, random_state=random.seed())
for train_index, test_index in kf.split(inputdata):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    # Set model parameters 
    Min_num_splits = ...                           # Minimum number of items to split
    Min_bucket     = ...                           # Minimum number of items per bucket
    Max_depth      = ...                           # Maximum depth of final tree (nr of levels)
    N_trees        = ...                           # Number of trees that are estimated
    Learning_rate  = ...                           # The learning rate (only applicable for Adaboost, XGBoost)
    Subsample      = ...                           # Subsample ratio of the training instance (only applicable for XGBoost)
    Silent         = ...                             # Whether to print messages while running boosting (only applicable for XGBoost)
    
    # a) Decision Tree CART
    j='Decision Tree'
    mytree = DecisionTreeClassifier(max_depth = Max_depth
                                 ,min_samples_split = Min_num_splits
                                 ,min_samples_leaf = Min_bucket
                                 ,criterion = "gini"              
                                 ,splitter = "best"
                                 ,random_state = random.seed()
                                 )
    mytree.fit(...)                    # Fit the model over the train set
    preds_proba = mytree.predict_proba(X_test)      # Calculate probabilities
    fpr, tpr, t = metrics.roc_curve(y_test, preds_proba[:, 1])  
    aucs.loc[i,j] = metrics.auc(fpr, tpr)           # Calculate AUC
    
    # b) Random Forest
    j='Random Forest'
    forest = RandomForestClassifier(n_estimators = N_trees
                               ,criterion = "gini"            
                               ,max_depth = Max_depth
                               ,min_samples_split = Min_num_splits
                               ,min_samples_leaf = Min_bucket
                               ,random_state = random.seed()
                               )
    forest.fit(...)                    # Fit the model over the train set
    preds_proba = ...      # Calculate probabilities
    fpr, tpr, t = ...  
    aucs.loc[i,j] = ...           # Calculate AUC
    
    # c) Adaboost
    j='Adaboost'
    adaptboost = AdaBoostClassifier(n_estimators = N_trees
                                    ,base_estimator = DecisionTreeClassifier(max_depth = Max_depth
                                                                             ,min_samples_split = Min_num_splits
                                                                             ,min_samples_leaf = Min_bucket
                                                                             ,criterion = "gini"              
                                                                             ,splitter = "best"
                                                                             )
                                    ,learning_rate = Learning_rate
                                    )
    adaptboost.fit(...)                # Fit the model over the train set
    preds_proba = ...  # Calculate probabilities
    fpr, tpr, t = ...  
    aucs.loc[i,j] = ...           # Calculate AUC
    
    # d) XGBoost
    j='XGBoost'
    XGB = xgb.XGBClassifier(n_estimators = N_trees
                            ,max_depth = Max_depth 
                            ,learning_rate = Learning_rate
                            ,min_child_weight = Min_bucket
                            ,subsample = Subsample
                            ,silent = Silent
                           )
    ...
    
    # Go to next iteration
    ...
    
# Evaluate the methods by looking at the average AUC
aucs.mean()    